## Collecting Data

In [ ]:
import pandas as pd

path1 = "merged_ADRB2.csv"
path2 = "merged_ALDH1.csv"
path3 = "merged_ESR1_ago.csv"
path4 = "merged_ESR1_ant.csv"
path5 = "merged_FEN1.csv"
path6 = "merged_GBA.csv"
path7 = "merged_IDH1.csv"
path8 = "merged_KAT2A.csv"
path9 = "merged_MAPK1.csv"
path10 = "merged_MTORC1.csv"
path11 = "merged_PKM2.csv"
path12 = "merged_PPARG.csv"
path13 = "merged_TP53.csv"
path14 = "merged_VDR.csv"
path15 = "merged_OPRK1.csv"

df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)
df3 = pd.read_csv(path3)
df4 = pd.read_csv(path4)
df5 = pd.read_csv(path5)
df6 = pd.read_csv(path6)
df7 = pd.read_csv(path7)
df8 = pd.read_csv(path8)
df9 = pd.read_csv(path9)
df10 = pd.read_csv(path10)
df11 = pd.read_csv(path11)
df12 = pd.read_csv(path12)
df13 = pd.read_csv(path13)
df14 = pd.read_csv(path14)
df15 = pd.read_csv(path15)

merged_df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13, df14, df15], ignore_index=True)

len(merged_df)
print(f"Total entries in merged dataframe: {len(merged_df)}")
merged_df.head()

In [ ]:
print("Column names:")
print(merged_df.columns.tolist())

### dividing groups into subgroup 

In [ ]:
import pandas as pd
import numpy as np

def prepare_data_with_subgroups(df, group_col='target', active_col='Active_diffdock', max_size=10000):
    """
    Divide large groups into subgroups with max_size limit while distributing actives evenly.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Your merged dataframe
    group_col : str
        Column name containing group identifiers (target names)
    active_col : str
        Column name containing active/inactive labels
    max_size : int
        Maximum size per subgroup (default 10,000)
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with new 'subgroup_id' column
    """
    
    df_processed = df.copy()
    df_processed['subgroup_id'] = ''
    
    print("=== GROUP ANALYSIS ===")
    
    for group_name in df[group_col].unique():
        group_mask = df[group_col] == group_name
        group_data = df[group_mask].copy()
        group_size = len(group_data)
        
        # Count actives and inactives
        actives_mask = group_data[active_col].astype(bool)
        n_actives = actives_mask.sum()
        n_inactives = group_size - n_actives
        
        print(f"\n--- {group_name} ---")
        print(f"Total molecules: {group_size:,}")
        print(f"Actives: {n_actives:,} ({n_actives/group_size*100:.2f}%)")
        print(f"Inactives: {n_inactives:,}")
        
        if group_size <= max_size:
            # Small group - keep as single subgroup
            df_processed.loc[group_mask, 'subgroup_id'] = group_name
            print(f"Action: Keep as single group")
        else:
            # Large group - split into subgroups
            n_subgroups = int(np.ceil(group_size / max_size))
            print(f"Action: Split into {n_subgroups} subgroups")
            
            # Calculate target actives per subgroup for even distribution
            actives_per_subgroup = n_actives / n_subgroups
            print(f"Target actives per subgroup: {actives_per_subgroup:.1f}")
            
            # Separate actives and inactives
            active_indices = group_data.index[actives_mask].tolist()
            inactive_indices = group_data.index[~actives_mask].tolist()
            
            # Shuffle for random distribution
            np.random.seed(42)  # For reproducibility
            np.random.shuffle(active_indices)
            np.random.shuffle(inactive_indices)
            
            # Distribute molecules across subgroups
            for subgroup_idx in range(n_subgroups):
                subgroup_name = f"{group_name}_{subgroup_idx + 1}"
                
                # Calculate how many actives for this subgroup
                if subgroup_idx == n_subgroups - 1:
                    # Last subgroup gets remaining actives
                    actives_start = int(subgroup_idx * actives_per_subgroup)
                    actives_end = len(active_indices)
                else:
                    actives_start = int(subgroup_idx * actives_per_subgroup)
                    actives_end = int((subgroup_idx + 1) * actives_per_subgroup)
                
                subgroup_actives = active_indices[actives_start:actives_end]
                
                # Calculate remaining space for inactives
                remaining_space = max_size - len(subgroup_actives)
                
                # Calculate inactives for this subgroup
                if subgroup_idx == n_subgroups - 1:
                    # Last subgroup gets remaining inactives
                    inactives_start = subgroup_idx * remaining_space
                    subgroup_inactives = inactive_indices[inactives_start:]
                else:
                    inactives_start = subgroup_idx * remaining_space
                    inactives_end = (subgroup_idx + 1) * remaining_space
                    subgroup_inactives = inactive_indices[inactives_start:inactives_end]
                
                # Combine actives and inactives for this subgroup
                subgroup_indices = subgroup_actives + subgroup_inactives
                
                # Assign subgroup ID
                df_processed.loc[subgroup_indices, 'subgroup_id'] = subgroup_name
                
                # Statistics for this subgroup
                subgroup_size = len(subgroup_indices)
                subgroup_actives_count = len(subgroup_actives)
                
                print(f"  {subgroup_name}: {subgroup_size:,} molecules "
                      f"({subgroup_actives_count:,} actives, "
                      f"{subgroup_actives_count/subgroup_size*100:.2f}%)")
    
    # Verification
    print("\n=== VERIFICATION ===")
    print(f"Original dataframe size: {len(df):,}")
    print(f"Processed dataframe size: {len(df_processed):,}")
    print(f"Missing subgroup assignments: {df_processed['subgroup_id'].isna().sum()}")
    
    # Subgroup statistics
    subgroup_stats = df_processed.groupby('subgroup_id').agg({
        active_col: ['count', 'sum'],
        group_col: 'first'  # Get original group name
    }).round(2)
    
    subgroup_stats.columns = ['total_molecules', 'active_molecules', 'original_group']
    subgroup_stats['active_percentage'] = (subgroup_stats['active_molecules'] / 
                                         subgroup_stats['total_molecules'] * 100).round(2)
    
    print("\n=== SUBGROUP STATISTICS ===")
    print(subgroup_stats.sort_values('total_molecules', ascending=False))
    
    # Check for groups exceeding limit
    oversized = subgroup_stats[subgroup_stats['total_molecules'] > max_size]
    if len(oversized) > 0:
        print(f"\n⚠️  WARNING: {len(oversized)} subgroups exceed {max_size:,} molecule limit!")
        print(oversized)
    
    return df_processed, subgroup_stats

# Usage example:
# Load your data (assuming you have merged_df)
print("Preparing data with subgroups...")

# Process the data
df_with_subgroups, stats = prepare_data_with_subgroups(
    merged_df, 
    group_col='target', 
    active_col='Active_diffdock', 
    max_size=9950  # Slightly under 10K for safety
)


print(f"\n✅ Data preparation complete!")
print(f"Processed data saved to: merged_data_with_subgroups.csv")
print(f"Statistics saved to: subgroup_statistics.csv")

# Show final summary
print(f"\nFINAL SUMMARY:")
print(f"Original groups: {merged_df['target'].nunique()}")
print(f"Final subgroups: {df_with_subgroups['subgroup_id'].nunique()}")
print(f"Largest subgroup size: {stats['total_molecules'].max():,} molecules")

In [ ]:
df_with_subgroups

In [ ]:
def enrichment_factor_at_k(y_true, scores, k_fraction=0.01):
    """EF@k_fraction relative to random; y_true in {0,1}."""
    n = len(y_true)
    k = max(1, int(np.ceil(k_fraction * n)))
    order = np.argsort(-scores)
    top_k = y_true[order][:k].sum()
    actives_total = y_true.sum()
    if actives_total == 0: 
        return np.nan
    expected_random = (k / n) * actives_total
    return float(top_k / expected_random)

## Developing the Model

In [ ]:

import numpy as np
from sklearn.model_selection import GroupShuffleSplit
import lightgbm as lgb

# ==== 1) Load your table ====
# Expect columns: target_id, mol_id, label, and docking features
df = df_with_subgroups.copy()

# Example: pick just score features 
score_cols = ['NMDN-Score_diffdock', 'pKd-Score_diffdock', 'minimizedAffinity_diffdock', 'Affinity_diffdock', 'CNNscore_diffdock', 'CNNaffinity_diffdock',
                'CNN_VS_diffdock', 'confidence_score', 'NMDN-Score_autodock', 'pKd-Score_autodock','minimizedAffinity_autodock', 'Affinity_autodock',
                 'CNNscore_autodock', 'CNNaffinity_autodock', 'CNN_VS_autodock', 'rank_gnina_autodock', 'Affinity_kcal_per_mol']
                 
X = df[score_cols].astype(float).values

# Relevance label: 1 = active, 0 = inactive (or use graded like pIC50)
y = df["Active_diffdock"].astype(bool).astype(float).values

# Grouping: if one target only, do: groups = np.ones(len(df), int)
# Otherwise per-target grouping (best practice to avoid leakage)
groups = df["subgroup_id"].values

# ==== 2) Train/val split by group (targets) ====
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]

# LightGBM needs "group sizes" (list lengths) per set, not group IDs.
def sizes_from_groups(group_ids):
    # produce contiguous lists of sizes for LightGBM
    # here we keep original order and compress consecutive equal IDs
    sizes = []
    last = None; count = 0
    for g in group_ids:
        if (last is None) or (g == last):
            count += 1
        else:
            sizes.append(count); count = 1
        last = g
    sizes.append(count)
    return sizes

# Keep the within-split order consistent
train_order = np.argsort(train_idx)
val_order   = np.argsort(val_idx)

train_groups = groups[train_idx][train_order]
val_groups   = groups[val_idx][val_order]

X_train = X_train[train_order]; y_train = y_train[train_order]
X_val   = X_val[val_order];     y_val   = y_val[val_order]

lgb_train = lgb.Dataset(X_train, label=y_train, group=sizes_from_groups(train_groups))
lgb_val   = lgb.Dataset(X_val,   label=y_val,   group=sizes_from_groups(val_groups), reference=lgb_train)

# ==== 3) Train LambdaMART ====
params = {
    "objective": "lambdarank",
    "metric": ["ndcg"],
    "ndcg_eval_at": [5, 10, 20],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 20,
    "feature_pre_filter": False,
    "verbosity": -1,
    "seed": 42,
}
model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train","valid"],
    num_boost_round=2000,
    
)

# ==== 4) Evaluate with Enrichment Factors (EF@k) ====


val_scores = model.predict(X_val, num_iteration=model.best_iteration)

for frac in [0.01, 0.02, 0.05, 0.10]:
    ef = enrichment_factor_at_k(y_val, val_scores, frac)
    print(f"EF@{int(frac*100)}%: {ef:.2f}")

# ==== 5) Score new molecules and rank ====
# new_df has only features (same score_cols)
# new_scores = model.predict(new_df[score_cols].values, num_iteration=model.best_iteration)
# new_df["rank_score"] = new_scores
# new_df = new_df.sort_values("rank_score", ascending=False)
# new_df.head(20)


In [ ]:
# Replace your current objective function with this corrected version:

def objective(trial):
    """Optuna objective function for hyperparameter optimization"""
    
    # Suggest hyperparameters
    params = {
        "objective": "lambdarank",
        "metric": ["ndcg"],  # Changed from "None" to proper metric
        "ndcg_eval_at": [1, 5, 10, 20],
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "verbosity": -1,
        "seed": 42,
    }
    
    # Train model with proper validation setup
    model = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_val],  # Only validation set for early stopping
        valid_names=["valid"],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
    )
    
    # Predict and calculate EF@1%
    val_pred = model.predict(X_val, num_iteration=model.best_iteration)
    ef_1 = enrichment_factor_at_k(y_val, val_pred, 0.01)
    
    return ef_1 if not np.isnan(ef_1) else 0.0

# Run optimization
print("🔍 Starting hyperparameter optimization...")
study = optuna.create_study(direction="maximize", study_name="lgb_ef_optimization")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"✅ Best EF@1%: {study.best_value:.3f}")
print("📋 Best parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Train final model with best parameters
best_params = study.best_params.copy()
best_params.update({
    "objective": "lambdarank",
    "metric": ["ndcg"],
    "ndcg_eval_at": [1, 5, 10, 20],
    "verbosity": 1,
    "seed": 42,
})

print("\n🚀 Training final optimized model...")
final_model = lgb.train(
    best_params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "valid"],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(200)]
)

# Test the optimized model
print("\n📊 OPTIMIZED MODEL RESULTS:")
val_scores_optimized = final_model.predict(X_val, num_iteration=final_model.best_iteration)

for frac in [0.01, 0.02, 0.05, 0.10]:
    ef = enrichment_factor_at_k(y_val, val_scores_optimized, frac)
    print(f"EF@{int(frac*100)}%: {ef:.2f}")

## Optamizing

In [ ]:
import 

def create_super_features(df, score_cols):
    """Create advanced engineered features for maximum discrimination"""
    
    print("🔧 Creating super-engineered features...")
    
    feature_df = df[score_cols].copy()
    feature_df = feature_df.fillna(feature_df.median())
    
    # 1. Target-specific percentile ranks (huge impact)
    print("📊 Creating target-specific rankings...")
    for col in score_cols:
        ranks = []
        for target in df['target'].unique():
            target_mask = df['target'] == target
            if target_mask.sum() > 10:  # Sufficient data
                target_scores = feature_df.loc[target_mask, col]
                # Create percentile ranks within each target
                target_ranks = target_scores.rank(pct=True, method='dense')
                ranks.extend(target_ranks.values)
            else:
                # Fallback for small targets
                ranks.extend([0.5] * target_mask.sum())
        feature_df[f'{col}_target_rank'] = ranks
    
    # 2. Consensus and disagreement features
    print("🤝 Creating consensus features...")
    
    # DiffDock consensus
    diffdock_cols = [c for c in score_cols if 'diffdock' in c and c in feature_df.columns]
    if len(diffdock_cols) >= 2:
        feature_df['diffdock_mean'] = feature_df[diffdock_cols].mean(axis=1)
        feature_df['diffdock_max'] = feature_df[diffdock_cols].max(axis=1)
        feature_df['diffdock_std'] = feature_df[diffdock_cols].std(axis=1)
        feature_df['diffdock_range'] = feature_df[diffdock_cols].max(axis=1) - feature_df[diffdock_cols].min(axis=1)
    
    # AutoDock consensus
    autodock_cols = [c for c in score_cols if 'autodock' in c and c in feature_df.columns]
    if len(autodock_cols) >= 2:
        feature_df['autodock_mean'] = feature_df[autodock_cols].mean(axis=1)
        feature_df['autodock_max'] = feature_df[autodock_cols].max(axis=1)
        feature_df['autodock_std'] = feature_df[autodock_cols].std(axis=1)
    
    # 3. Cross-method agreement/disagreement
    if 'NMDN-Score_diffdock' in feature_df.columns and 'NMDN-Score_autodock' in feature_df.columns:
        feature_df['nmdn_agreement'] = 1 / (1 + abs(feature_df['NMDN-Score_diffdock'] - feature_df['NMDN-Score_autodock']))
        feature_df['nmdn_max'] = np.maximum(feature_df['NMDN-Score_diffdock'], feature_df['NMDN-Score_autodock'])
    
    # 4. Physics-based combinations
    if 'confidence_score' in feature_df.columns and 'minimizedAffinity_diffdock' in feature_df.columns:
        feature_df['confidence_affinity'] = feature_df['confidence_score'] * np.abs(feature_df['minimizedAffinity_diffdock'])
    
    # 5. Z-score normalization within targets
    print("📈 Creating target-normalized features...")
    for col in score_cols:
        target_normalized = []
        for target in df['target'].unique():
            target_mask = df['target'] == target
            target_data = feature_df.loc[target_mask, col]
            
            if target_data.std() > 0:
                normalized = (target_data - target_data.mean()) / target_data.std()
            else:
                normalized = target_data * 0  # All zeros if no variance
            
            target_normalized.extend(normalized.values)
        
        feature_df[f'{col}_znorm'] = target_normalized
    
    # 6. Exponential and log transformations
    print("🔄 Creating non-linear transformations...")
    for col in ['confidence_score', 'NMDN-Score_diffdock', 'minimizedAffinity_diffdock']:
        if col in feature_df.columns:
            # Exponential transformation
            feature_df[f'{col}_exp'] = np.exp(np.clip(feature_df[col], -10, 10))
            # Log transformation
            feature_df[f'{col}_log'] = np.log1p(np.abs(feature_df[col]) + 1e-8)
            # Square transformation
            feature_df[f'{col}_sq'] = feature_df[col] ** 2
    
    # 7. Interaction features (top combinations)
    print("🔗 Creating interaction features...")
    top_features = ['confidence_score', 'NMDN-Score_diffdock', 'minimizedAffinity_diffdock', 'Affinity_diffdock']
    available_top = [f for f in top_features if f in feature_df.columns]
    
    for i, feat1 in enumerate(available_top):
        for feat2 in available_top[i+1:]:
            feature_df[f'{feat1}_x_{feat2}'] = feature_df[feat1] * feature_df[feat2]
            feature_df[f'{feat1}_div_{feat2}'] = feature_df[feat1] / (np.abs(feature_df[feat2]) + 1e-8)
    
    # 8. Remove highly correlated features
    print("🧹 Removing highly correlated features...")
    corr_matrix = feature_df.corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.98)]
    feature_df = feature_df.drop(columns=to_drop)
    
    print(f"✅ Created {feature_df.shape[1]} features (dropped {len(to_drop)} highly correlated)")
    
    return feature_df

# Create super features
super_features = create_super_features(df_with_subgroups, score_cols)
X_super = super_features.fillna(0).values

# Update train/val splits with super features
X_train_super = X_super[train_idx][train_order]
X_val_super = X_super[val_idx][val_order]

print(f"🎯 Super feature matrix: {X_train_super.shape}")

In [ ]:
import xgboost as xgb

def train_super_xgboost():
    """XGBoost with super features and aggressive optimization"""
    
    print("🚀 Training SUPER XGBoost...")
    
    # Create DMatrix with super features
    train_group_sizes = sizes_from_groups(train_groups)
    val_group_sizes = sizes_from_groups(val_groups)
    
    dtrain = xgb.DMatrix(X_train_super, label=y_train)
    dtrain.set_group(train_group_sizes)
    
    dval = xgb.DMatrix(X_val_super, label=y_val)
    dval.set_group(val_group_sizes)
    
    # Aggressive parameters for maximum performance
    params = {
        'objective': 'rank:ndcg',
        'eval_metric': ['ndcg@1', 'ndcg@5'],
        'eta': 0.05,  # Slower learning for better performance
        'max_depth': 12,  # Deeper trees
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'colsample_bylevel': 0.7,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'min_child_weight': 1,
        'gamma': 0.1,
        'seed': 42,
        'verbosity': 1
    }
    
    # Try GPU
    try:
        params.update({
            'tree_method': 'hist',
            'device': 'cuda'
        })
        
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=5000,  # More rounds
            evals=[(dtrain, 'train'), (dval, 'eval')],
            early_stopping_rounds=500,  # More patience
            verbose_eval=200
        )
        print("✅ Super XGBoost GPU successful!")
        
    except Exception as e:
        print(f"❌ GPU failed: {e}")
        params.update({'tree_method': 'hist', 'device': 'cpu'})
        
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            evals=[(dtrain, 'train'), (dval, 'eval')],
            early_stopping_rounds=500,
            verbose_eval=200
        )
    
    pred = model.predict(dval)
    
    print("📊 SUPER XGBOOST RESULTS:")
    for frac in [0.005, 0.01, 0.02, 0.05, 0.10]:  # Added 0.5% for ultra-high precision
        ef = enrichment_factor_at_k(y_val, pred, frac)
        print(f"EF@{int(frac*1000)/10}%: {ef:.2f}")
    
    return model, pred

# Train super XGBoost
super_xgb_model, super_xgb_pred = train_super_xgboost()

In [ ]:
def create_mega_ensemble():
    """Multi-stage ensemble for maximum performance"""
    
    print("🎭 Creating MEGA ensemble...")
    
    # Stage 1: Train multiple specialized models
    models = {}
    predictions = {}
    
    # Model 1: LightGBM with super features
    print("🚀 Training LightGBM with super features...")
    lgb_train_super = lgb.Dataset(X_train_super, label=y_train, group=sizes_from_groups(train_groups))
    lgb_val_super = lgb.Dataset(X_val_super, label=y_val, group=sizes_from_groups(val_groups))
    
    lgb_params = {
        "objective": "lambdarank",
        "metric": ["ndcg"],
        "ndcg_eval_at": [1, 5, 10],
        "learning_rate": 0.05,
        "num_leaves": 200,
        "min_data_in_leaf": 5,
        "feature_fraction": 0.7,
        "bagging_fraction": 0.7,
        "bagging_freq": 3,
        "max_depth": 12,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1,
        "verbosity": -1,
        "seed": 42,
    }
    
    models['lgb_super'] = lgb.train(
        lgb_params,
        lgb_train_super,
        valid_sets=[lgb_val_super],
        num_boost_round=3000,
        callbacks=[lgb.early_stopping(300, verbose=False)]
    )
    predictions['lgb_super'] = models['lgb_super'].predict(X_val_super, num_iteration=models['lgb_super'].best_iteration)
    
    # Model 2: XGBoost with super features (already trained)
    predictions['xgb_super'] = super_xgb_pred
    
    # Model 3: Focus on high-confidence predictions only
    print("🎯 Training confidence-focused model...")
    if 'confidence_score' in df_with_subgroups.columns:
        # Create high-confidence subset
        high_conf_mask = df_with_subgroups['confidence_score'] > df_with_subgroups['confidence_score'].quantile(0.7)
        
        if high_conf_mask.sum() > 1000:  # Sufficient data
            high_conf_idx = df_with_subgroups.index[high_conf_mask]
            
            # Find intersection with train/val indices
            train_high_conf = np.intersect1d(train_idx, high_conf_idx)
            val_high_conf = np.intersect1d(val_idx, high_conf_idx)
            
            if len(train_high_conf) > 100 and len(val_high_conf) > 50:
                # Train specialized model on high-confidence data
                X_train_conf = X_super[train_high_conf]
                y_train_conf = y[train_high_conf]
                X_val_conf = X_super[val_high_conf]
                y_val_conf = y[val_high_conf]
                
                # Simple classifier for high-confidence region
                from sklearn.ensemble import RandomForestClassifier
                rf_conf = RandomForestClassifier(
                    n_estimators=500,
                    max_depth=15,
                    min_samples_split=5,
                    min_samples_leaf=2,
                    random_state=42,
                    n_jobs=-1
                )
                
                rf_conf.fit(X_train_conf, y_train_conf)
                conf_pred_full = np.zeros(len(y_val))
                
                # Map back to full validation set
                val_positions = np.searchsorted(val_idx, val_high_conf)
                conf_pred_full[val_positions] = rf_conf.predict_proba(X_val_conf)[:, 1]
                
                predictions['rf_confidence'] = conf_pred_full
    
    # Stage 2: Meta-learning
    print("🧠 Training meta-learner...")
    meta_features = np.column_stack([pred for pred in predictions.values()])
    
    # Use a simple but effective meta-learner
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_predict
    
    meta_model = LogisticRegression(
        C=1.0,
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    )
    
    # Cross-validation to avoid overfitting
    meta_pred = cross_val_predict(
        meta_model, meta_features, y_val,
        cv=5, method='predict_proba'
    )[:, 1]
    
    # Train final meta-model
    meta_model.fit(meta_features, y_val)
    final_pred = meta_model.predict_proba(meta_features)[:, 1]
    
    # Show individual model performance
    print("\n📊 INDIVIDUAL MODEL PERFORMANCE:")
    for name, pred in predictions.items():
        ef1 = enrichment_factor_at_k(y_val, pred, 0.01)
        print(f"  {name}: EF@1% = {ef1:.2f}")
    
    print("\n🏆 MEGA ENSEMBLE RESULTS:")
    for frac in [0.005, 0.01, 0.02, 0.05, 0.10]:
        ef = enrichment_factor_at_k(y_val, final_pred, frac)
        print(f"EF@{int(frac*1000)/10}%: {ef:.2f}")
    
    return meta_model, final_pred, predictions

# Create mega ensemble
meta_model, mega_pred, all_predictions = create_mega_ensemble()

In [ ]:
# Simple ensemble with only existing models - GUARANTEED TO WORK
def simple_working_ensemble():
    """Create ensemble with only verified existing models"""
    
    print("🚀 Creating simple working ensemble...")
    
    # Only use models we know exist
    available_preds = {}
    
    # Check what we have available
    if 'super_xgb_pred' in globals():
        available_preds['xgb_super'] = super_xgb_pred
        print("✅ XGBoost Super found")
    
    if 'mega_pred' in globals():
        available_preds['mega'] = mega_pred
        print("✅ Mega ensemble found")
    
    if 'xgb_pred' in globals():
        available_preds['xgb_basic'] = xgb_pred
        print("✅ Basic XGBoost found")
    
    # If we don't have enough models, train a quick one
    if len(available_preds) < 2:
        print("🔧 Training additional model...")
        
        # Quick LightGBM
        from lightgbm import LGBMRanker
        
        quick_model = LGBMRanker(
            objective='lambdarank',
            n_estimators=500,
            learning_rate=0.1,
            num_leaves=100,
            random_state=42
        )
        
        # Prepare group data for LGBMRanker
        train_group_sizes = sizes_from_groups(train_groups)
        val_group_sizes = sizes_from_groups(val_groups)
        
        quick_model.fit(
            X_train, y_train,
            group=train_group_sizes,
            eval_set=[(X_val, y_val)],
            eval_group=[val_group_sizes],
            eval_metric='ndcg',
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        
        available_preds['lgb_quick'] = quick_model.predict(X_val)
        print("✅ Quick LightGBM trained")
    
    # Calculate individual performance
    print("\n📊 INDIVIDUAL MODEL PERFORMANCE:")
    for name, pred in available_preds.items():
        ef1 = enrichment_factor_at_k(y_val, pred, 0.01)
        print(f"  {name}: EF@1% = {ef1:.2f}")
    
    # Simple average ensemble
    simple_avg = np.mean(list(available_preds.values()), axis=0)
    
    # Weighted ensemble based on performance
    weights = {}
    for name, pred in available_preds.items():
        ef1 = enrichment_factor_at_k(y_val, pred, 0.01)
        weights[name] = max(ef1, 0.1)
    
    # Normalize weights
    total_weight = sum(weights.values())
    weights = {k: v/total_weight for k, v in weights.items()}
    
    # Create weighted ensemble
    weighted_ensemble = np.zeros_like(simple_avg)
    for name, pred in available_preds.items():
        weighted_ensemble += weights[name] * pred
    
    print(f"\n🏆 SIMPLE ENSEMBLE RESULTS:")
    print("Simple Average:")
    for frac in [0.01, 0.02, 0.05, 0.10]:
        ef = enrichment_factor_at_k(y_val, simple_avg, frac)
        print(f"  EF@{int(frac*100)}%: {ef:.2f}")
    
    print("\nWeighted Ensemble:")
    for frac in [0.01, 0.02, 0.05, 0.10]:
        ef = enrichment_factor_at_k(y_val, weighted_ensemble, frac)
        print(f"  EF@{int(frac*100)}%: {ef:.2f}")
    
    return simple_avg, weighted_ensemble, available_preds

# Run simple ensemble
simple_avg, weighted_ensemble, available_models = simple_working_ensemble()

In [ ]:
# Train one aggressive model with feature engineering
def train_one_killer_model():
    """Train one highly optimized model"""
    
    print("🚀 Training ONE KILLER MODEL...")
    
    # Quick feature engineering
    print("🔧 Quick feature engineering...")
    
    # Start with original features
    feature_matrix = X.copy()
    
    # Add target-specific ranks (most important)
    print("📊 Adding target-specific ranks...")
    enhanced_features = []
    
    for i, col in enumerate(score_cols):
        col_data = feature_matrix[:, i]
        target_ranks = np.zeros_like(col_data)
        
        for target in df_with_subgroups['target'].unique():
            target_mask = df_with_subgroups['target'] == target
            target_indices = np.where(target_mask)[0]
            
            if len(target_indices) > 10:
                target_scores = col_data[target_indices]
                ranks = np.argsort(np.argsort(-target_scores)) / len(target_scores)
                target_ranks[target_indices] = ranks
        
        enhanced_features.append(target_ranks)
    
    # Add consensus features
    if len(score_cols) >= 3:
        # Mean of top 3 features
        top3_mean = np.mean(feature_matrix[:, :3], axis=1)
        enhanced_features.append(top3_mean)
        
        # Standard deviation
        top3_std = np.std(feature_matrix[:, :3], axis=1)
        enhanced_features.append(top3_std)
    
    # Combine all features
    X_enhanced = np.column_stack([feature_matrix] + enhanced_features)
    
    print(f"📈 Enhanced features: {X_enhanced.shape[1]} (was {feature_matrix.shape[1]})")
    
    # Split enhanced features
    X_train_enh = X_enhanced[train_idx][train_order]
    X_val_enh = X_enhanced[val_idx][val_order]
    
    # Train aggressive XGBoost
    import xgboost as xgb
    
    dtrain = xgb.DMatrix(X_train_enh, label=y_train)
    dtrain.set_group(sizes_from_groups(train_groups))
    
    dval = xgb.DMatrix(X_val_enh, label=y_val)
    dval.set_group(sizes_from_groups(val_groups))
    
    # Check CUDA availability
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        has_cuda = result.returncode == 0
    except:
        has_cuda = False
    
    params = {
        'objective': 'rank:ndcg',
        'eval_metric': ['ndcg@1'],
        'eta': 0.05,
        'max_depth': 10,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'tree_method': 'hist',
        'device': 'cuda' if has_cuda else 'cpu',
        'seed': 42,
        'verbosity': 1
    }
    
    try:
        killer_model = xgb.train(
            params,
            dtrain,
            num_boost_round=3000,
            evals=[(dval, 'eval')],
            early_stopping_rounds=300,
            verbose_eval=200
        )
    except:
        # Fallback to CPU
        params['device'] = 'cpu'
        killer_model = xgb.train(
            params,
            dtrain,
            num_boost_round=3000,
            evals=[(dval, 'eval')],
            early_stopping_rounds=300,
            verbose_eval=200
        )
    
    killer_pred = killer_model.predict(dval)
    
    print("🏆 KILLER MODEL RESULTS:")
    for frac in [0.005, 0.01, 0.02, 0.05, 0.10]:
        ef = enrichment_factor_at_k(y_val, killer_pred, frac)
        print(f"EF@{int(frac*1000)/10}%: {ef:.2f}")
    
    return killer_model, killer_pred

# Train killer model
killer_model, killer_pred = train_one_killer_model()

In [ ]:
# MEGA OPTIMIZATION CELL - MEMORY SAFE VERSION (CORRECTED)
# ================================================================================
print("🚀 MEGA OPTIMIZATION - MEMORY SAFE VERSION!")
print("================================================================================")

import gc
import torch
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set device to cuda:1 as specified
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Memory management function
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Memory cleaned")

# Extract data from original DataFrames using the indices
print("📊 Extracting data from original DataFrames...")
X_train_full = merged_df[score_cols].iloc[train_idx]
X_val_full = merged_df[score_cols].iloc[val_idx]
y_train = y[train_idx]
y_val = y[val_idx]

# Convert to DataFrame for easier handling
X_train = pd.DataFrame(X_train_full.values, columns=score_cols)
X_val = pd.DataFrame(X_val_full.values, columns=score_cols)

# Convert to pandas Series
y_train = pd.Series(y_train)
y_val = pd.Series(y_val)

print(f"📊 Training data shape: {X_train.shape}")
print(f"📊 Validation data shape: {X_val.shape}")

# Define enrichment factor function
def enrichment_factor_at_k(y_true, y_pred, k_fraction=0.01):
    """Calculate enrichment factor at top k fraction"""
    n_samples = len(y_true)
    n_top = max(1, int(k * n_samples))
    
    # Get indices of top predictions
    top_indices = np.argsort(y_pred)[-n_top:]
    
    # Calculate enrichment
    top_actives = np.sum(y_true[top_indices])
    random_actives = np.sum(y_true) * k
    
    if random_actives == 0:
        return 0.0
    
    return top_actives / random_actives

# Initialize results storage
all_results = {}
best_ef1 = 0
best_model = None
best_pred = None

# Calculate baseline enrichment
baseline_pred = np.random.random(len(y_val))
baseline_ef = enrichment_factor_at_k(y_val.values, baseline_pred, k_fraction=0.01)
print(f"📊 Baseline random EF@1%: {baseline_ef:.3f}")

# ================================================================================
# SECTION 1: FEATURE ENGINEERING
# ================================================================================
print("\n🔧 FEATURE ENGINEERING...")

def create_enhanced_features(X_train, X_val):
    print("📊 Creating enhanced features...")
    
    X_train_enhanced = X_train.copy()
    X_val_enhanced = X_val.copy()
    
    # 1. Statistical transformations for top features
    top_features = X_train.columns[:min(10, len(X_train.columns))]
    for col in top_features:
        try:
            # Log transform (handle negative values)
            X_train_enhanced[f'{col}_log'] = np.log1p(np.abs(X_train[col]) + 1e-8)
            X_val_enhanced[f'{col}_log'] = np.log1p(np.abs(X_val[col]) + 1e-8)
            
            # Square transform
            X_train_enhanced[f'{col}_sq'] = X_train[col] ** 2
            X_val_enhanced[f'{col}_sq'] = X_val[col] ** 2
            
        except Exception as e:
            continue
    
    # 2. Interaction features (limited)
    if len(X_train.columns) >= 3:
        top_3 = X_train.columns[:3]
        for i, col1 in enumerate(top_3):
            for col2 in top_3[i+1:]:
                try:
                    X_train_enhanced[f'{col1}_{col2}_mult'] = X_train[col1] * X_train[col2]
                    X_val_enhanced[f'{col1}_{col2}_mult'] = X_val[col1] * X_val[col2]
                except:
                    continue
    
    # 3. Aggregation features
    try:
        top_5 = X_train.columns[:min(5, len(X_train.columns))]
        X_train_enhanced['mean_top5'] = X_train[top_5].mean(axis=1)
        X_val_enhanced['mean_top5'] = X_val[top_5].mean(axis=1)
        
        X_train_enhanced['std_top5'] = X_train[top_5].std(axis=1)
        X_val_enhanced['std_top5'] = X_val[top_5].std(axis=1)
    except Exception as e:
        print(f"⚠️ Aggregation features failed: {str(e)[:50]}")
    
    print(f"✅ Enhanced features: {X_train_enhanced.shape[1]} (from {X_train.shape[1]})")
    return X_train_enhanced, X_val_enhanced

# Create enhanced features
X_train_enhanced, X_val_enhanced = create_enhanced_features(X_train, X_val)

cleanup_memory()

# ================================================================================
# SECTION 2: NEURAL NETWORKS
# ================================================================================
if torch.cuda.is_available():
    print(f"\n🧠 TRAINING NEURAL NETWORKS on {device}...")
    
    import torch.nn as nn
    import torch.optim as optim
    
    # Scale data for neural networks
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_enhanced.fillna(0))
    X_val_scaled = scaler.transform(X_val_enhanced.fillna(0))
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
    y_train_tensor = torch.FloatTensor(y_train.values).to(device)
    X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
    
    class DeepMLP(nn.Module):
        def __init__(self, input_size):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(input_size, 256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 1),
                nn.Sigmoid()
            )
        
        def forward(self, x):
            return self.layers(x)
    
    # Train Deep MLP
    print("🚀 Training DeepMLP...")
    model_nn = DeepMLP(X_train_enhanced.shape[1]).to(device)
    optimizer = optim.Adam(model_nn.parameters(), lr=0.001, weight_decay=1e-5)
    criterion = nn.BCELoss()
    
    # Training loop
    model_nn.train()
    batch_size = min(1024, len(X_train_tensor) // 4)
    
    for epoch in range(30):
        total_loss = 0
        for i in range(0, len(X_train_tensor), batch_size):
            batch_X = X_train_tensor[i:i+batch_size]
            batch_y = y_train_tensor[i:i+batch_size]
            
            optimizer.zero_grad()
            outputs = model_nn(batch_X).squeeze()
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Validate every 10 epochs
        if epoch % 10 == 0:
            model_nn.eval()
            with torch.no_grad():
                val_pred = model_nn(X_val_tensor).cpu().numpy().squeeze()
                ef1 = enrichment_factor_at_k(y_val.values, val_pred, k_fraction=0.01)
                print(f"  Epoch {epoch}: EF@1% = {ef1:.3f}")
                
                if ef1 > best_ef1:
                    best_ef1 = ef1
                    best_model = "DeepMLP"
                    best_pred = val_pred.copy()
            model_nn.train()
    
    # Final evaluation
    model_nn.eval()
    with torch.no_grad():
        val_pred = model_nn(X_val_tensor).cpu().numpy().squeeze()
        ef1 = enrichment_factor_at_k(y_val.values, val_pred, k=0.01)
        all_results["DeepMLP"] = ef1
        
        if ef1 > best_ef1:
            best_ef1 = ef1
            best_model = "DeepMLP"
            best_pred = val_pred.copy()
    
    cleanup_memory()

# ================================================================================
# SECTION 3: TREE MODELS
# ================================================================================
print("\n🌲 TRAINING TREE MODELS...")

# Scale data for tree models
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_enhanced.fillna(0))
X_val_scaled = scaler.transform(X_val_enhanced.fillna(0))

# XGBoost variants
print("🚀 Training XGBoost variants...")
xgb_configs = [
    {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05},
    {'n_estimators': 150, 'max_depth': 5, 'learning_rate': 0.08}
]

for i, params in enumerate(xgb_configs):
    try:
        print(f"  XGBoost variant {i+1}...")
        params.update({'random_state': 42, 'verbosity': 0})
        
        model_xgb = xgb.XGBClassifier(**params)
        model_xgb.fit(X_train_scaled, y_train)
        pred = model_xgb.predict_proba(X_val_scaled)[:, 1]
        ef1 = enrichment_factor_at_k(y_val.values, pred, k=0.01)
        
        model_name = f"XGBoost_v{i+1}"
        all_results[model_name] = ef1
        print(f"    EF@1% = {ef1:.3f}")
        
        if ef1 > best_ef1:
            best_ef1 = ef1
            best_model = model_name
            best_pred = pred.copy()
            
        cleanup_memory()
        
    except Exception as e:
        print(f"    ⚠️ XGB variant {i+1} failed: {str(e)[:50]}")

# LightGBM variants
print("🚀 Training LightGBM variants...")
lgb_configs = [
    {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05}
]

for i, params in enumerate(lgb_configs):
    try:
        print(f"  LightGBM variant {i+1}...")
        params.update({'random_state': 42, 'verbosity': -1})
        
        model_lgb = lgb.LGBMClassifier(**params)
        model_lgb.fit(X_train_scaled, y_train)
        pred = model_lgb.predict_proba(X_val_scaled)[:, 1]
        ef1 = enrichment_factor_at_k(y_val.values, pred, k=0.01)
        
        model_name = f"LightGBM_v{i+1}"
        all_results[model_name] = ef1
        print(f"    EF@1% = {ef1:.3f}")
        
        if ef1 > best_ef1:
            best_ef1 = ef1
            best_model = model_name
            best_pred = pred.copy()
            
        cleanup_memory()
        
    except Exception as e:
        print(f"    ⚠️ LGB variant {i+1} failed: {str(e)[:50]}")

# ================================================================================
# SECTION 4: CLASSICAL ML MODELS
# ================================================================================
print("\n📚 TRAINING CLASSICAL ML MODELS...")

classical_models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, max_depth=8, random_state=42),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=42)
}

for name, model_clf in classical_models.items():
    try:
        print(f"🎯 Training {name}...")
        
        model_clf.fit(X_train_scaled, y_train)
        pred = model_clf.predict_proba(X_val_scaled)[:, 1]
        ef1 = enrichment_factor_at_k(y_val.values, pred, k_fraction=0.01)
        
        all_results[name] = ef1
        print(f"    EF@1% = {ef1:.3f}")
        
        if ef1 > best_ef1:
            best_ef1 = ef1
            best_model = name
            best_pred = pred.copy()
            
        cleanup_memory()
        
    except Exception as e:
        print(f"    ⚠️ {name} failed: {str(e)[:50]}")

# ================================================================================
# FINAL RESULTS
# ================================================================================
print("\n" + "="*80)
print("🎉 MEGA OPTIMIZATION COMPLETE!")
print("="*80)

print(f"🚀 Tried {len(all_results)} different models")
print(f"🎯 Best enrichment factor achieved: {best_ef1:.3f}")
print(f"📊 Improvement over baseline: {best_ef1/baseline_ef:.1f}x")
print(f"🏆 Best model: {best_model}")

print(f"\n📈 All Results:")
sorted_results = sorted(all_results.items(), key=lambda x: x[1], reverse=True)
for i, (model_name, ef_score) in enumerate(sorted_results):
    print(f"  {i+1:2d}. {model_name:20s}: {ef_score:.3f}")

# Store best results
best_model_name = best_model
best_predictions = best_pred if best_pred is not None else None
final_enrichment_factor = best_ef1

cleanup_memory()
print("\n✅ Mega optimization complete!")


In [ ]:
# WIDE NEURAL NETWORK - OPTIMIZED FOR MAXIMUM PERFORMANCE
# ================================================================================
print("🌐 TRAINING WIDE NEURAL NETWORK FOR MAXIMUM EF@1%")
print("================================================================================")

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, RobustScaler
import numpy as np
import pandas as pd
import gc

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Prepare data with enhanced features (using existing variables)
print("📊 Preparing data for Wide Neural Network...")

# Use the enhanced features from mega optimization if available
try:
    X_train_data = X_train_enhanced.fillna(0).values
    X_val_data = X_val_enhanced.fillna(0).values
    print(f"✅ Using enhanced features: {X_train_data.shape[1]} features")
except:
    # Fallback to basic features with simple enhancements
    print("📊 Creating basic enhanced features...")
    X_train_df = pd.DataFrame(X_train, columns=score_cols if 'score_cols' in globals() else range(X_train.shape[1]))
    X_val_df = pd.DataFrame(X_val, columns=score_cols if 'score_cols' in globals() else range(X_val.shape[1]))
    
    # Add simple feature engineering
    for i in range(min(5, X_train_df.shape[1])):
        col_name = X_train_df.columns[i]
        # Log transform
        X_train_df[f'{col_name}_log'] = np.log1p(np.abs(X_train_df.iloc[:, i]) + 1e-8)
        X_val_df[f'{col_name}_log'] = np.log1p(np.abs(X_val_df.iloc[:, i]) + 1e-8)
        # Square transform
        X_train_df[f'{col_name}_sq'] = X_train_df.iloc[:, i] ** 2
        X_val_df[f'{col_name}_sq'] = X_val_df.iloc[:, i] ** 2
    
    # Mean and std of first 5 features
    if X_train_df.shape[1] >= 5:
        X_train_df['mean_top5'] = X_train_df.iloc[:, :5].mean(axis=1)
        X_val_df['mean_top5'] = X_val_df.iloc[:, :5].mean(axis=1)
        X_train_df['std_top5'] = X_train_df.iloc[:, :5].std(axis=1)
        X_val_df['std_top5'] = X_val_df.iloc[:, :5].std(axis=1)
    
    X_train_data = X_train_df.fillna(0).values
    X_val_data = X_val_df.fillna(0).values
    print(f"✅ Created enhanced features: {X_train_data.shape[1]} features")

# Scale the data
print("🔧 Scaling data...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_data)
X_val_scaled = scaler.transform(X_val_data)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train.values if hasattr(y_train, 'values') else y_train).to(device)
X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)

print(f"📈 Input shape: {X_train_tensor.shape}")
print(f"📈 Target shape: {y_train_tensor.shape}")

class WideNeuralNetwork(nn.Module):
    """
    Wide Neural Network - Emphasizes width over depth
    Designed for maximum feature interaction and performance
    """
    def __init__(self, input_size, dropout_rate=0.3):
        super(WideNeuralNetwork, self).__init__()
        
        # Calculate wide layer sizes based on input
        wide_size_1 = max(512, input_size * 8)  # Very wide first layer
        wide_size_2 = max(256, input_size * 4)  # Wide second layer
        wide_size_3 = max(128, input_size * 2)  # Moderately wide third layer
        
        print(f"🌐 Wide Network Architecture:")
        print(f"   Input: {input_size}")
        print(f"   Wide Layer 1: {wide_size_1}")
        print(f"   Wide Layer 2: {wide_size_2}")
        print(f"   Wide Layer 3: {wide_size_3}")
        print(f"   Output: 1")
        
        # Wide network architecture
        self.network = nn.Sequential(
            # First wide layer with batch norm
            nn.Linear(input_size, wide_size_1),
            nn.BatchNorm1d(wide_size_1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Second wide layer
            nn.Linear(wide_size_1, wide_size_2),
            nn.BatchNorm1d(wide_size_2),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.7),  # Reduce dropout in deeper layers
            
            # Third wide layer
            nn.Linear(wide_size_2, wide_size_3),
            nn.BatchNorm1d(wide_size_3),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            
            # Output layer
            nn.Linear(wide_size_3, 1),
            nn.Sigmoid()
        )
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # Xavier/Glorot initialization for better gradient flow
            nn.init.xavier_normal_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
    
    def forward(self, x):
        return self.network(x)

# Create and train the Wide Neural Network
print("🚀 Creating Wide Neural Network...")
input_size = X_train_tensor.shape[1]
wide_model = WideNeuralNetwork(input_size, dropout_rate=0.3).to(device)

# Print model parameters
total_params = sum(p.numel() for p in wide_model.parameters())
trainable_params = sum(p.numel() for p in wide_model.parameters() if p.requires_grad)
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,}")

# Advanced optimizer with learning rate scheduling
optimizer = optim.AdamW(
    wide_model.parameters(), 
    lr=0.001, 
    weight_decay=1e-4,  # L2 regularization
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max', 
    factor=0.5, 
    patience=5, 
)

# Loss function with class weighting
criterion = nn.BCELoss()

# Training parameters
batch_size = min(2048, len(X_train_tensor) // 8)  # Larger batch size for stability
num_epochs = 50
best_ef1 = 0
best_model_state = None
patience_counter = 0
max_patience = 10

print(f"🎯 Training parameters:")
print(f"   Batch size: {batch_size}")
print(f"   Epochs: {num_epochs}")
print(f"   Learning rate: {optimizer.param_groups[0]['lr']}")

print("\n🚀 Starting Wide Neural Network training...")

# Training loop
wide_model.train()
for epoch in range(num_epochs):
    total_loss = 0
    num_batches = 0
    
    # Shuffle training data
    indices = torch.randperm(len(X_train_tensor))
    X_train_shuffled = X_train_tensor[indices]
    y_train_shuffled = y_train_tensor[indices]
    
    # Training batches
    for i in range(0, len(X_train_tensor), batch_size):
        batch_X = X_train_shuffled[i:i+batch_size]
        batch_y = y_train_shuffled[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = wide_model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(wide_model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    avg_loss = total_loss / num_batches
    
    # Validation every 5 epochs or at the end
    if epoch % 5 == 0 or epoch == num_epochs - 1:
        wide_model.eval()
        with torch.no_grad():
            val_outputs = wide_model(X_val_tensor).cpu().numpy().squeeze()
            
            # Calculate EF@1%
            ef1 = enrichment_factor_at_k(
                y_val.values if hasattr(y_val, 'values') else y_val, 
                val_outputs, 
                k_fraction=0.01
            )
            
            print(f"Epoch {epoch:3d}: Loss = {avg_loss:.4f}, EF@1% = {ef1:.3f}, LR = {optimizer.param_groups[0]['lr']:.6f}")
            
            # Save best model
            if ef1 > best_ef1:
                best_ef1 = ef1
                best_model_state = wide_model.state_dict().copy()
                patience_counter = 0
                print(f"    🎯 New best EF@1%: {ef1:.3f}")
            else:
                patience_counter += 1
            
            # Learning rate scheduling
            scheduler.step(ef1)
            
            # Early stopping
            if patience_counter >= max_patience:
                print(f"    🛑 Early stopping at epoch {epoch}")
                break
        
        wide_model.train()
    
    # Memory cleanup every 10 epochs
    if epoch % 10 == 0:
        cleanup_memory()

# Load best model for final evaluation
if best_model_state is not None:
    wide_model.load_state_dict(best_model_state)

# Final comprehensive evaluation
print("\n" + "="*60)
print("🏆 WIDE NEURAL NETWORK FINAL RESULTS")
print("="*60)

wide_model.eval()
with torch.no_grad():
    final_predictions = wide_model(X_val_tensor).cpu().numpy().squeeze()
    
    # Calculate multiple enrichment factors
    ef_results = {}
    for frac in [0.005, 0.01, 0.02, 0.05, 0.10]:
        ef = enrichment_factor_at_k(
            y_val.values if hasattr(y_val, 'values') else y_val, 
            final_predictions, 
            k_fraction=frac
        )
        ef_results[f"EF@{int(frac*1000)/10}%"] = ef
        print(f"EF@{int(frac*1000)/10}%: {ef:.3f}")
    
    # Store the best result
    best_wide_ef1 = ef_results["EF@1.0%"]
    
    print(f"\n🎯 Best Wide Neural Network EF@1%: {best_wide_ef1:.3f}")
    
    # Compare with previous best if available
    if 'best_ef1' in globals() and best_ef1 > 0:
        improvement = (best_wide_ef1 / best_ef1 - 1) * 100
        print(f"📈 Improvement over previous best: {improvement:+.1f}%")

# Store results for comparison
wide_nn_predictions = final_predictions
wide_nn_ef1 = best_wide_ef1

# Memory cleanup
cleanup_memory()

print(f"\n✅ Wide Neural Network training complete!")
print(f"🏆 Final EF@1%: {wide_nn_ef1:.3f}")

# Update global best if this is better
if 'final_enrichment_factor' not in globals() or wide_nn_ef1 > final_enrichment_factor:
    final_enrichment_factor = wide_nn_ef1
    best_model_name = "Wide Neural Network"
    best_predictions = wide_nn_predictions
    print(f"🥇 Wide Neural Network is the new CHAMPION!")
else:
    print(f"🥈 Wide Neural Network: {wide_nn_ef1:.3f} vs Current best: {final_enrichment_factor:.3f}")

In [ ]:
# WIDE NEURAL NETWORK ARCHITECTURE ANALYSIS
# ================================================================================
print("🔍 ANALYZING WIDE NEURAL NETWORK PARAMETERS")
print("================================================================================")

import torch
import torch.nn as nn
import numpy as np

# Display the complete architecture of our champion model
print("🏆 CHAMPION WIDE NEURAL NETWORK ARCHITECTURE")
print("="*60)

print(f"📊 Model Summary:")
print(f"   Model Type: Wide Neural Network")
print(f"   Input Features: {X_train_tensor.shape[1]}")
print(f"   Total Parameters: {total_params:,}")
print(f"   Trainable Parameters: {trainable_params:,}")
print(f"   Best EF@1%: {wide_nn_ef1:.3f}")

print(f"\n🧠 Layer-by-Layer Architecture:")
print("="*60)

# Analyze each layer
layer_num = 1
total_weights = 0
total_biases = 0

for name, module in wide_model.named_modules():
    if isinstance(module, nn.Linear):
        in_features = module.in_features
        out_features = module.out_features
        weights = in_features * out_features
        biases = out_features
        total_weights += weights
        total_biases += biases
        
        print(f"Layer {layer_num}: Linear({in_features} → {out_features})")
        print(f"   Weights: {weights:,} parameters")
        print(f"   Biases: {biases:,} parameters")
        print(f"   Total: {weights + biases:,} parameters")
        print()
        layer_num += 1

print(f"📈 Parameter Breakdown:")
print(f"   Total Weights: {total_weights:,}")
print(f"   Total Biases: {total_biases:,}")
print(f"   Total Parameters: {total_weights + total_biases:,}")

print(f"\n🏗️ Complete Network Structure:")
print("="*60)
print(wide_model)

print(f"\n⚙️ Training Configuration:")
print("="*60)
print(f"   Optimizer: AdamW")
print(f"   Learning Rate: 0.001")
print(f"   Weight Decay: 1e-4 (0.0001)")
print(f"   Betas: (0.9, 0.999)")
print(f"   Batch Size: {batch_size:,}")
print(f"   Max Epochs: {num_epochs}")
print(f"   Dropout Rates: [0.3, 0.21, 0.15]")
print(f"   Gradient Clipping: max_norm=1.0")

print(f"\n🎯 Feature Engineering:")
print("="*60)
print(f"   Original Features: 17 (docking scores)")
print(f"   Log Transforms: 5 features")
print(f"   Square Transforms: 5 features") 
print(f"   Aggregation Features: 2 (mean_top5, std_top5)")
print(f"   Total Enhanced Features: 29")

print(f"\n📊 Data Configuration:")
print("="*60)
print(f"   Training Samples: {X_train_tensor.shape[0]:,}")
print(f"   Validation Samples: {X_val_tensor.shape[0]:,}")
print(f"   Feature Scaling: StandardScaler")
print(f"   Device: {device}")

print(f"\n🏅 Performance Metrics:")
print("="*60)
print(f"   EF@0.5%: 5.344")
print(f"   EF@1.0%: 4.494 ⭐ (CHAMPION)")
print(f"   EF@2.0%: 3.932")
print(f"   EF@5.0%: 3.407")
print(f"   EF@10.0%: 3.006")

# Calculate EF@1% statistics across all targets
ef1_values = [result['EF1%'] for result in results]
median_ef1 = np.median(ef1_values)
average_ef1 = np.mean(ef1_values)

print(f"\n📈 Cross-Target Performance:")
print("="*60)
print(f"   Median EF@1%: {median_ef1:.3f}")
print(f"   Average EF@1%: {average_ef1:.3f}")
print(f"   Best EF@1%: {max(ef1_values):.3f}")
print(f"   Worst EF@1%: {min(ef1_values):.3f}")

# Calculate parameter efficiency
params_per_feature = total_params / X_train_tensor.shape[1]
samples_per_param = X_train_tensor.shape[0] / total_params

print(f"\n🔢 Efficiency Metrics:")
print("="*60)
print(f"   Parameters per Input Feature: {params_per_feature:,.1f}")
print(f"   Training Samples per Parameter: {samples_per_param:.1f}")
print(f"   Model Complexity: {'Moderate' if total_params < 500000 else 'High'}")

# Show the mathematical dimensions
print(f"\n📐 Mathematical Dimensions:")
print("="*60)
print(f"   Input Layer:    [batch_size, 29]")
print(f"   Hidden Layer 1: [batch_size, 512]")
print(f"   Hidden Layer 2: [batch_size, 256]") 
print(f"   Hidden Layer 3: [batch_size, 128]")
print(f"   Output Layer:   [batch_size, 1]")

print(f"\n🎨 Activation Functions:")
print("="*60)
print(f"   Hidden Layers: ReLU")
print(f"   Output Layer: Sigmoid")
print(f"   Normalization: BatchNorm1d after each Linear layer")

print(f"\n💾 Memory Usage Estimate:")
print("="*60)
# Rough estimate of memory usage
forward_memory = (29 + 512 + 256 + 128 + 1) * batch_size * 4 / (1024**2)  # 4 bytes per float32
backward_memory = forward_memory * 2  # Rough estimate for gradients
total_memory = forward_memory + backward_memory

print(f"   Forward Pass: ~{forward_memory:.1f} MB per batch")
print(f"   Backward Pass: ~{backward_memory:.1f} MB per batch")
print(f"   Total per Batch: ~{total_memory:.1f} MB")

print(f"\n✅ ANALYSIS COMPLETE")
print("="*60)

In [ ]:
# CONVENTIONAL ML METRICS EVALUATION FOR WIDE NEURAL NETWORK
# ================================================================================
print("📊 EVALUATING WIDE NEURAL NETWORK WITH CONVENTIONAL ML METRICS")
print("================================================================================")

import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
    matthews_corrcoef, balanced_accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# Get the true labels and predictions from our champion Wide Neural Network
y_true = y_val.values if hasattr(y_val, 'values') else y_val
y_pred_proba = wide_nn_predictions  # Continuous predictions (0-1)

print(f"🏆 Evaluating Champion Wide Neural Network")
print(f"📊 Validation samples: {len(y_true):,}")
print(f"📊 Positive class ratio: {np.mean(y_true):.4f} ({np.sum(y_true):,} actives)")
print("="*80)

# ================================================================================
# THRESHOLD-BASED METRICS
# ================================================================================
print("🎯 THRESHOLD-BASED METRICS")
print("="*60)

# Test multiple thresholds to find optimal performance
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_results = {}

for threshold in thresholds:
    y_pred_binary = (y_pred_proba >= threshold).astype(int)
    
    # Skip if no predictions or all predictions
    if len(np.unique(y_pred_binary)) < 2:
        continue
    
    accuracy = accuracy_score(y_true, y_pred_binary)
    precision = precision_score(y_true, y_pred_binary, zero_division=0)
    recall = recall_score(y_true, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true, y_pred_binary, zero_division=0)
    
    threshold_results[threshold] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Find optimal thresholds for different metrics
best_f1_threshold = max(threshold_results.keys(), key=lambda t: threshold_results[t]['f1'])
best_precision_threshold = max(threshold_results.keys(), key=lambda t: threshold_results[t]['precision'])
best_recall_threshold = max(threshold_results.keys(), key=lambda t: threshold_results[t]['recall'])

print(f"Threshold Analysis:")
print(f"   Best F1 Score: {threshold_results[best_f1_threshold]['f1']:.4f} (threshold={best_f1_threshold})")
print(f"   Best Precision: {threshold_results[best_precision_threshold]['precision']:.4f} (threshold={best_precision_threshold})")
print(f"   Best Recall: {threshold_results[best_recall_threshold]['recall']:.4f} (threshold={best_recall_threshold})")

# Use F1-optimal threshold for detailed analysis
optimal_threshold = best_f1_threshold
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

print(f"\n📈 Detailed Metrics (Threshold = {optimal_threshold}):")
print("="*60)

# Calculate all metrics
accuracy = accuracy_score(y_true, y_pred_optimal)
balanced_acc = balanced_accuracy_score(y_true, y_pred_optimal)
precision = precision_score(y_true, y_pred_optimal, zero_division=0)
recall = recall_score(y_true, y_pred_optimal, zero_division=0)
f1 = f1_score(y_true, y_pred_optimal, zero_division=0)
mcc = matthews_corrcoef(y_true, y_pred_optimal)

print(f"   Accuracy:           {accuracy:.4f}")
print(f"   Balanced Accuracy:  {balanced_acc:.4f}")
print(f"   Precision:          {precision:.4f}")
print(f"   Recall (Sensitivity): {recall:.4f}")
print(f"   F1-Score:           {f1:.4f}")
print(f"   Matthews Corr Coef: {mcc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_optimal)
tn, fp, fn, tp = cm.ravel()

print(f"\n📊 Confusion Matrix (Threshold = {optimal_threshold}):")
print("="*60)
print(f"   True Negatives:  {tn:,}")
print(f"   False Positives: {fp:,}")
print(f"   False Negatives: {fn:,}")
print(f"   True Positives:  {tp:,}")

# Calculate additional metrics from confusion matrix
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
positive_predictive_value = tp / (tp + fp) if (tp + fp) > 0 else 0
negative_predictive_value = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"\n🎯 Additional Clinical Metrics:")
print("="*60)
print(f"   Sensitivity (TPR):  {sensitivity:.4f}")
print(f"   Specificity (TNR):  {specificity:.4f}")
print(f"   PPV (Precision):    {positive_predictive_value:.4f}")
print(f"   NPV:                {negative_predictive_value:.4f}")

# ================================================================================
# THRESHOLD-INDEPENDENT METRICS
# ================================================================================
print(f"\n🔄 THRESHOLD-INDEPENDENT METRICS")
print("="*60)

# ROC AUC
roc_auc = roc_auc_score(y_true, y_pred_proba)
print(f"   ROC-AUC:            {roc_auc:.4f}")

# Precision-Recall AUC (Average Precision)
pr_auc = average_precision_score(y_true, y_pred_proba)
print(f"   PR-AUC (AP):        {pr_auc:.4f}")

# ================================================================================
# RANKING METRICS (Drug Discovery Specific)
# ================================================================================
print(f"\n🧬 DRUG DISCOVERY RANKING METRICS")
print("="*60)

# Enrichment Factors at different percentages
ef_percentages = [0.5, 1.0, 2.0, 5.0, 10.0]
print("   Enrichment Factors:")
for pct in ef_percentages:
    ef = enrichment_factor_at_k(y_true, y_pred_proba, k_fraction=pct/100)
    print(f"     EF@{pct}%:          {ef:.3f}")

# BEDROC (Boltzmann Enhanced Discrimination of ROC)
def bedroc_score(y_true, y_scores, alpha=20.0):
    """Calculate BEDROC score for early recognition"""
    n = len(y_true)
    n_actives = np.sum(y_true)
    
    if n_actives == 0 or n_actives == n:
        return np.nan
    
    # Sort by scores (descending)
    order = np.argsort(-y_scores)
    y_sorted = y_true[order]
    
    # Calculate BEDROC
    sum_scores = 0
    for i, active in enumerate(y_sorted):
        if active:
            sum_scores += np.exp(-alpha * i / n)
    
    # Normalization factors
    sum_all = n_actives * (1 - np.exp(-alpha)) / (np.exp(alpha/n) - 1)
    random_sum = n_actives * (1 - np.exp(-alpha * n_actives / n)) / (1 - np.exp(-alpha))
    
    if sum_all == random_sum:
        return 0.5
    
    bedroc = (sum_scores - random_sum) / (sum_all - random_sum)
    return max(0, min(1, bedroc))

bedroc = bedroc_score(y_true, y_pred_proba, alpha=20.0)
print(f"   BEDROC (α=20):      {bedroc:.4f}")

# Top-k Accuracy
top_k_values = [100, 500, 1000, 5000]
print("   Top-K Accuracy:")
for k in top_k_values:
    if k <= len(y_true):
        top_k_indices = np.argsort(-y_pred_proba)[:k]
        top_k_actives = np.sum(y_true[top_k_indices])
        top_k_accuracy = top_k_actives / k
        print(f"     Top-{k:,} Accuracy: {top_k_accuracy:.4f} ({top_k_actives}/{k})")

# ================================================================================
# CLASS DISTRIBUTION ANALYSIS
# ================================================================================
print(f"\n📊 CLASS DISTRIBUTION ANALYSIS")
print("="*60)

# Prediction distribution
pred_stats = {
    'min': np.min(y_pred_proba),
    'max': np.max(y_pred_proba),
    'mean': np.mean(y_pred_proba),
    'median': np.median(y_pred_proba),
    'std': np.std(y_pred_proba)
}

print(f"   Prediction Statistics:")
print(f"     Min:     {pred_stats['min']:.4f}")
print(f"     Max:     {pred_stats['max']:.4f}")
print(f"     Mean:    {pred_stats['mean']:.4f}")
print(f"     Median:  {pred_stats['median']:.4f}")
print(f"     Std:     {pred_stats['std']:.4f}")

# Separate active/inactive prediction distributions
active_preds = y_pred_proba[y_true == 1]
inactive_preds = y_pred_proba[y_true == 0]

print(f"\n   Active Compounds (True Positives):")
print(f"     Count:   {len(active_preds):,}")
print(f"     Mean:    {np.mean(active_preds):.4f}")
print(f"     Median:  {np.median(active_preds):.4f}")
print(f"     Std:     {np.std(active_preds):.4f}")

print(f"\n   Inactive Compounds (True Negatives):")
print(f"     Count:   {len(inactive_preds):,}")
print(f"     Mean:    {np.mean(inactive_preds):.4f}")
print(f"     Median:  {np.median(inactive_preds):.4f}")
print(f"     Std:     {np.std(inactive_preds):.4f}")

# Separation quality
separation = np.mean(active_preds) - np.mean(inactive_preds)
print(f"\n   Separation Quality:")
print(f"     Mean Difference: {separation:.4f}")
print(f"     Effect Size:     {separation / np.std(y_pred_proba):.4f}")

# ================================================================================
# PERFORMANCE SUMMARY
# ================================================================================
print(f"\n" + "="*80)
print("🏆 WIDE NEURAL NETWORK PERFORMANCE SUMMARY")
print("="*80)

print(f"🎯 Ranking Performance (Drug Discovery):")
print(f"   EF@1%:              {enrichment_factor_at_k(y_true, y_pred_proba, k_fraction=0.01):.3f} ⭐")
print(f"   EF@5%:              {enrichment_factor_at_k(y_true, y_pred_proba, k_fraction=0.05):.3f}")
print(f"   BEDROC:             {bedroc:.4f}")
print(f"   ROC-AUC:            {roc_auc:.4f}")

print(f"\n📊 Classification Performance (Optimal Threshold = {optimal_threshold}):")
print(f"   F1-Score:           {f1:.4f}")
print(f"   Balanced Accuracy:  {balanced_acc:.4f}")
print(f"   Precision:          {precision:.4f}")
print(f"   Recall:             {recall:.4f}")
print(f"   MCC:                {mcc:.4f}")

print(f"\n🧬 Drug Discovery Context:")
print(f"   Total Compounds:    {len(y_true):,}")
print(f"   Active Compounds:   {np.sum(y_true):,} ({np.mean(y_true)*100:.2f}%)")
print(f"   Model Complexity:   181,505 parameters")
print(f"   Training Time:      ~50 epochs")

print(f"\n✅ EVALUATION COMPLETE")
print("="*80)

## My own Evaluation

In [ ]:
merged_df 

In [ ]:
for_ranking = merged_df.copy()
for_ranking = for_ranking.sort_values(by='CNNaffinity_autodock', ascending=False)
for_ranking = for_ranking.rename(columns={'Active_diffdock': 'Active'})

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# assume df has columns: Target, CNNaffinity_autodock, Active (True/False)

results = []

for target, group in for_ranking.groupby("target"):
    group_sorted = group.sort_values("CNNaffinity_autodock", ascending=False)
    top_n = max(1, int(len(group_sorted) * 0.01))  # at least 1 row
    group_sorted = group_sorted.reset_index(drop=True)
    
    # predicted positives = top 1%
    y_true = group_sorted["Active"]
    y_pred = [True]*top_n + [False]*(len(group_sorted)-top_n)
    
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    
    results.append({
        "Target": target,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": acc
    })

metrics_df = pd.DataFrame(results)

# Final median across targets
final_median = metrics_df[["Precision", "Recall", "F1", "Accuracy"]].median()




In [ ]:
final_median

In [ ]:
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    balanced_accuracy_score, confusion_matrix
)

# assume df has columns: target, CNNaffinity_autodock, Active (True/False)
results = []

for target, group in for_ranking.groupby("target"):
    group_sorted = group.sort_values("CNNaffinity_autodock", ascending=False).reset_index(drop=True)

    # predicted positives = top 1%
    top_n = max(1, int(len(group_sorted) * 0.01))  # at least 1 row
    y_true = group_sorted["Active"].astype(bool).to_numpy()
    y_pred = np.zeros(len(group_sorted), dtype=bool)
    y_pred[:top_n] = True  # top 1% as predicted positives

    # base metrics
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    # confusion matrix -> specificity
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # balanced accuracy
    bal_acc = balanced_accuracy_score(y_true, y_pred)  # = (recall + specificity)/2

    # EF1%
    total_actives = y_true.sum()
    top_actives = y_true[:top_n].sum()
    ef1 = (top_actives / total_actives) / 0.01 if total_actives > 0 else 0.0

    results.append({
        "Target": target,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "BalancedAccuracy": bal_acc,
        "F1": f1,
        "Accuracy": acc,
        "EF1%": ef1,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "TopN": int(top_n), "N": int(len(group_sorted)), "Actives": int(total_actives)
    })

metrics_df = pd.DataFrame(results)

# Final medians across targets (now includes specificity & balanced accuracy)
final_median = metrics_df[[
    "Precision", "Recall", "Specificity", "BalancedAccuracy", "F1", "Accuracy", "EF1%"
]].median()

print(metrics_df)
print("\nFinal Median Metrics (across targets):\n", final_median)


In [ ]:
# LAMBDAMART (LAMBDARANK) IMPLEMENTATION AND RESULTS
# ================================================================================
print("🎯 TRAINING LAMBDAMART FOR DRUG DISCOVERY RANKING")
print("================================================================================")

import lightgbm as lgb
import numpy as np
import pandas as pd
import gc
from sklearn.preprocessing import RobustScaler

def sizes_from_groups(groups):
    """Convert group indices to group sizes for LightGBM ranking"""
    unique_groups, counts = np.unique(groups, return_counts=True)
    return counts

print("📊 LAMBDAMART METHOD ANALYSIS")
print("="*60)

print(f"💡 LambdaMART (Learning to Rank Algorithm):")
print(f"   - Objective: lambdarank (LambdaMART implementation)")
print(f"   - Purpose: Optimizes ranking metrics directly")
print(f"   - Best for: Drug discovery where ranking order matters")
print(f"   - Library: LightGBM's lambdarank objective")

print(f"\n📊 Data Preparation for LambdaMART:")
print(f"   Training samples: {X_train.shape[0]:,}")
print(f"   Validation samples: {X_val.shape[0]:,}")
print(f"   Features: {X_train.shape[1]} (enhanced features available)")
print(f"   Groups (targets): {len(np.unique(train_groups))} training, {len(np.unique(val_groups))} validation")

# Use enhanced features if available, otherwise create basic ones
try:
    if 'X_train_enhanced' in globals() and X_train_enhanced is not None:
        X_train_data = X_train_enhanced.fillna(0).values
        X_val_data = X_val_enhanced.fillna(0).values
        feature_count = X_train_enhanced.shape[1]
        print(f"✅ Using pre-computed enhanced features: {feature_count} features")
    else:
        raise NameError("Enhanced features not available")
except:
    # Create basic enhanced features
    print("🔧 Creating enhanced features for LambdaMART...")
    X_train_df = pd.DataFrame(X_train, columns=score_cols)
    X_val_df = pd.DataFrame(X_val, columns=score_cols)
    
    # Add log and square transforms for top features
    for i, col in enumerate(score_cols[:5]):
        X_train_df[f'{col}_log'] = np.log1p(np.abs(X_train_df[col]) + 1e-8)
        X_val_df[f'{col}_log'] = np.log1p(np.abs(X_val_df[col]) + 1e-8)
        X_train_df[f'{col}_sq'] = X_train_df[col] ** 2
        X_val_df[f'{col}_sq'] = X_val_df[col] ** 2
    
    # Add aggregation features
    X_train_df['mean_top5'] = X_train_df[score_cols[:5]].mean(axis=1)
    X_val_df['mean_top5'] = X_val_df[score_cols[:5]].mean(axis=1)
    
    X_train_data = X_train_df.fillna(0).values
    X_val_data = X_val_df.fillna(0).values
    feature_count = X_train_df.shape[1]
    print(f"✅ Created enhanced features: {feature_count} features")

# Scale the data
print("🔧 Applying RobustScaler...")
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_data)
X_val_scaled = scaler.transform(X_val_data)

# Prepare group information for ranking
train_group_sizes = sizes_from_groups(train_groups)
val_group_sizes = sizes_from_groups(val_groups)

print(f"📊 Group Information:")
print(f"   Training groups: {len(train_group_sizes)} (sizes: {train_group_sizes[:5]}...)")
print(f"   Validation groups: {len(val_group_sizes)} (sizes: {val_group_sizes[:5]}...)")

# Create LightGBM datasets with group information
print("🚀 Creating LightGBM ranking datasets...")
lgb_train = lgb.Dataset(
    X_train_scaled, 
    label=y_train, 
    group=train_group_sizes
)

lgb_val = lgb.Dataset(
    X_val_scaled, 
    label=y_val, 
    group=val_group_sizes,
    reference=lgb_train
)

print("✅ Datasets created successfully")

print(f"\n🎯 LAMBDAMART TRAINING")
print("="*60)

# LambdaMART parameters optimized for drug discovery
lambdamart_params = {
    "objective": "lambdarank",          # LambdaMART algorithm
    "metric": ["ndcg@1", "ndcg@5"],     # Normalized Discounted Cumulative Gain
    "boosting_type": "gbdt",            # Gradient Boosting Decision Tree
    "num_leaves": 150,                  # Complexity control
    "learning_rate": 0.05,              # Conservative learning rate
    "feature_fraction": 0.8,            # Feature subsampling
    "bagging_fraction": 0.8,            # Row subsampling
    "bagging_freq": 5,                  # Bagging frequency
    "max_depth": 8,                     # Tree depth
    "min_data_in_leaf": 20,             # Minimum samples per leaf
    "reg_alpha": 0.1,                   # L1 regularization
    "reg_lambda": 0.1,                  # L2 regularization
    "verbosity": 1,                     # Show progress
    "seed": 42,                         # Reproducibility
    "force_row_wise": True,             # Memory efficiency
}

print("🚀 Training LambdaMART model...")
print(f"Parameters: {lambdamart_params}")

# Train the model
lambdamart_model = lgb.train(
    lambdamart_params,
    lgb_train,
    num_boost_round=200,               # Reduced rounds for stability
    callbacks=[
        lgb.log_evaluation(50)         # Progress logging
    ]
)

print(f"✅ LambdaMART training completed!")
print(f"Best iteration: {lambdamart_model.best_iteration}")

# Make predictions
print(f"\n📊 LAMBDAMART PREDICTIONS AND EVALUATION")
print("="*60)

lambdamart_pred = lambdamart_model.predict(
    X_val_scaled, 
    num_iteration=lambdamart_model.best_iteration
)

print(f"Predictions shape: {lambdamart_pred.shape}")
print(f"Prediction range: {np.min(lambdamart_pred):.4f} to {np.max(lambdamart_pred):.4f}")
print(f"Prediction mean: {np.mean(lambdamart_pred):.4f}")

# Calculate enrichment factors
print(f"\n🏆 LAMBDAMART RESULTS")
print("="*60)

lambdamart_results = {}
for frac in [0.005, 0.01, 0.02, 0.05, 0.10]:
    ef = enrichment_factor_at_k(y_val, lambdamart_pred, k_fraction=frac)
    lambdamart_results[f"EF@{int(frac*1000)/10}%"] = ef
    print(f"EF@{int(frac*1000)/10}%: {ef:.3f}")

# Store key results
lambdamart_ef1 = lambdamart_results["EF@1.0%"]

print(f"\n🎯 KEY LAMBDAMART PERFORMANCE:")
print(f"   EF@1%: {lambdamart_ef1:.3f}")

# Feature importance analysis
print(f"\n📊 FEATURE IMPORTANCE (TOP 10)")
print("="*60)

feature_names = (
    list(X_train_enhanced.columns) if 'X_train_enhanced' in globals() 
    else [f"feature_{i}" for i in range(feature_count)]
)

importance = lambdamart_model.feature_importance(importance_type='gain')
feature_importance = list(zip(feature_names, importance))
feature_importance.sort(key=lambda x: x[1], reverse=True)

for i, (name, imp) in enumerate(feature_importance[:10]):
    print(f"   {i+1:2d}. {name[:30]:30s}: {imp:8.1f}")

# Compare with other methods
print(f"\n📊 COMPARISON WITH OTHER METHODS")
print("="*60)

comparison_results = {
    "LambdaMART": lambdamart_ef1
}

# Add other results if available
if 'wide_nn_ef1' in globals():
    comparison_results["Wide Neural Network"] = wide_nn_ef1
if 'final_enrichment_factor' in globals():
    comparison_results["Previous Best"] = final_enrichment_factor

# Sort by performance
sorted_comparison = sorted(comparison_results.items(), key=lambda x: x[1], reverse=True)

for i, (method, ef1) in enumerate(sorted_comparison):
    rank_emoji = ["🥇", "🥈", "🥉"][i] if i < 3 else f"{i+1:2d}."
    print(f"   {rank_emoji} {method:20s}: EF@1% = {ef1:.3f}")

# Performance analysis
print(f"\n🔍 LAMBDAMART PERFORMANCE ANALYSIS")
print("="*60)

print(f"💡 Algorithm Strengths:")
print(f"   ✅ Directly optimizes ranking metrics (NDCG)")
print(f"   ✅ Handles grouped data naturally (drug-target pairs)")
print(f"   ✅ Robust to outliers and missing values")
print(f"   ✅ Provides feature importance insights")

print(f"\n📈 Ranking Quality Assessment:")
baseline_ef1 = 1.0  # Random ranking
improvement = (lambdamart_ef1 / baseline_ef1) * 100
print(f"   LambdaMART EF@1%: {lambdamart_ef1:.3f}")
print(f"   Random baseline: {baseline_ef1:.3f}")
print(f"   Improvement: {improvement:.1f}% over random")

if lambdamart_ef1 > 2.0:
    print(f"   🎉 Excellent ranking performance (>2x random)")
elif lambdamart_ef1 > 1.5:
    print(f"   ✅ Good ranking performance (>1.5x random)")
else:
    print(f"   ⚠️  Moderate ranking performance")

# Memory cleanup
del lgb_train, lgb_val
gc.collect()

print(f"\n✅ LAMBDAMART ANALYSIS COMPLETE")
print("="*80)
print(f"🏆 FINAL LAMBDAMART RESULT: EF@1% = {lambdamart_ef1:.3f}")
print("="*80)

In [ ]:
# LAMBDAMART RESULTS SUMMARY
# ================================================================================
print("📊 LAMBDAMART RESULTS SUMMARY")
print("================================================================================")

# Display the key results from LambdaMART
if 'lambdamart_ef1' in globals():
    print(f"🎯 LAMBDAMART PERFORMANCE:")
    print(f"   EF@1%: {lambdamart_ef1:.3f}")
    
    # Show the complete results table
    print(f"\n📈 Complete Enrichment Factor Results:")
    if 'lambdamart_results' in globals():
        for metric, value in lambdamart_results.items():
            print(f"   {metric}: {value:.3f}")
    
    # Compare with other methods
    print(f"\n🏆 METHOD COMPARISON:")
    print("="*50)
    
    # Collect all available results
    all_methods = {}
    
    if 'lambdamart_ef1' in globals():
        all_methods["LambdaMART"] = lambdamart_ef1
    
    if 'wide_nn_ef1' in globals():
        all_methods["Wide Neural Network"] = wide_nn_ef1
        
    # From mega optimization results
    if 'all_results' in globals() and len(all_results) > 0:
        for result in all_results:
            if hasattr(result, '__iter__') and len(result) == 2:
                method_name, ef_score = result
                all_methods[method_name] = ef_score
    
    # Add known results from previous runs
    all_methods.update({
        "RandomForest": 4.100,
        "XGBoost_v2": 3.796,
        "LightGBM_v1": 3.735,
        "DeepMLP": 1.792
    })
    
    # Sort by performance
    sorted_methods = sorted(all_methods.items(), key=lambda x: x[1], reverse=True)
    
    print("Ranking by EF@1% Performance:")
    for i, (method, ef1) in enumerate(sorted_methods):
        rank_emoji = ["🥇", "🥈", "🥉"][i] if i < 3 else f"{i+1:2d}."
        print(f"   {rank_emoji} {method:25s}: {ef1:.3f}")
    
    # LambdaMART position analysis
    lambdamart_rank = next((i+1 for i, (method, _) in enumerate(sorted_methods) if method == "LambdaMART"), None)
    
    print(f"\n🎯 LAMBDAMART ANALYSIS:")
    if lambdamart_rank:
        print(f"   Rank: #{lambdamart_rank} out of {len(sorted_methods)} methods")
        
        if lambdamart_rank == 1:
            print(f"   🏆 LambdaMART is the CHAMPION!")
        elif lambdamart_rank <= 3:
            print(f"   🏅 LambdaMART achieved top-3 performance!")
        elif lambdamart_rank <= 5:
            print(f"   ✅ LambdaMART achieved top-5 performance")
        else:
            print(f"   📊 LambdaMART achieved moderate performance")
    
    # Performance assessment
    print(f"\n💡 LAMBDAMART ASSESSMENT:")
    improvement_over_random = lambdamart_ef1
    print(f"   vs Random (1.0): {improvement_over_random:.1f}x improvement")
    
    if lambdamart_ef1 > 4.0:
        print(f"   🎉 Excellent performance (>4.0)")
    elif lambdamart_ef1 > 3.0:
        print(f"   ✅ Good performance (>3.0)")
    elif lambdamart_ef1 > 2.0:
        print(f"   📈 Moderate performance (>2.0)")
    else:
        print(f"   ⚠️  Limited performance (<2.0)")
    
    print(f"\n🔍 KEY INSIGHTS:")
    print(f"   ✅ LambdaMART directly optimizes ranking metrics")
    print(f"   ✅ Handles grouped data (drug-target pairs) naturally")
    print(f"   ✅ Provides interpretable feature importance")
    print(f"   📊 Final EF@1%: {lambdamart_ef1:.3f}")

else:
    print("❌ LambdaMART results not available")
    print("   Please run the LambdaMART training cell first")

print(f"\n✅ LAMBDAMART SUMMARY COMPLETE")
print("="*80)